# Evaluation Confidence Scoring Methodology

This notebook documents the chunk-based confidence methodology used to assess how reliable an evaluation judgment is, given the evidence supplied.

The goal is to **separate conformance probability from confidence of judgment**:
- **P(conformant)**: how likely the requirement is satisfied based on evidence.
- **Confidence**: how reliable that judgment is, considering coverage, contradictions, missingness, and robustness to chunking.

The approach is designed for audit workflows where missing evidence is a strong negative signal and where chunk boundaries introduce uncertainty.


## Evidence Variables

For each requirement, decompose it into **rubric items** (sub-claims). For each chunk and item, label the relation:

- **support**: direct evidence that satisfies the item
- **weak**: partial or vague support
- **contradict**: explicit conflict with the item
- **irrelevant**: unrelated text
- **insufficient**: mentions the topic but lacks required specifics

Optional metadata on each label (when available):
- **specificity** (1–5): how precise and testable the evidence is
- **directness** (1–5): how directly the chunk supports the item

From these labels we compute per-finding features:

- **COV** (coverage): fraction of rubric items with support or weak support
- **SUP** (support strength): weighted support mass (support > weak), scaled by specificity
- **CONTRA**: weighted contradiction mass
- **MISS**: number of rubric items with no support
- **MISS_RATIO**: MISS divided by total items
- **SPEC**: average specificity of supporting labels
- **CONS** (consistency): penalty if an item has both support and contradiction
- **ROB** (robustness): stability across chunk-boundary perturbations or approximation


## Bayesian-ish Scoring Model

Each rubric item *i* is modeled as a Bernoulli variable (satisfied or not). We use a Beta prior and update with evidence:

- Prior: **Beta(α0, β0)**
- Evidence updates:
  - support adds to **α**
  - contradict adds to **β**
  - weak adds smaller α
  - missingness adds a **β** penalty

Per-item mean:

\[E[p_i] = \frac{\alpha_i}{\alpha_i + \beta_i}\]

Requirement-level conformance probability (all items required) is the geometric mean:

\[P(\text{conformant}) = \exp\left(\frac{1}{k}\sum_i \log E[p_i]\right)\]

**Confidence of judgment** is derived from evidence quality rather than from the probability alone. We use a weighted feature score:

\[\text{confidence} = \sigma(w \cdot [COV, SUP, CONTRA, MISS, SPEC, CONS, ROB])\]

Then, robustness shrinks confidence toward 0.5 if perturbations are unstable.

### Robustness approximation (default)
To avoid extra LLM calls, robustness is estimated from structural indicators derived from the label matrix:

- fraction of supported items backed by only one chunk
- weak-to-strong support ratio
- contradiction density
- redundancy of support (average number of supporting chunks)
- edge-chunk reliance (chunks at boundary indices)

These indicators are combined into a penalty and mapped to a robustness score in \([0,1]\).

### Chunk impact scoring
For each (finding, chunk), we estimate contribution without re-evaluation:

1. For each rubric item *i*, count supporting chunks \(n_i\)
2. Assign shared credit \(\text{credit}(c,i)=\frac{w_i}{n_i}\) when chunk \(c\) supports item \(i\)
3. Multiply by chunk-level **specificity** and **directness**
4. Normalize across all chunks so impacts sum to 1

This yields a scalar impact score that answers: "How much did this chunk matter to this finding?"


In [ ]:
import math


def posterior_probability(support_mass, contra_mass, alpha0=1.0, beta0=1.0, missing_penalty=1.0):
    # Single-item example
    alpha = alpha0 + support_mass
    beta = beta0 + contra_mass
    if support_mass == 0 and contra_mass == 0:
        beta += missing_penalty
    return alpha / (alpha + beta)


def confidence_score(cov, sup, contra, miss_ratio, spec, cons, rob=1.0):
    score = (
        1.8 * cov
        + 1.2 * sup
        - 1.5 * contra
        - 1.0 * miss_ratio
        + 0.8 * spec
        + 0.6 * cons
        + 0.5 * rob
        - 1.0
    )
    conf = 1.0 / (1.0 + math.exp(-score))
    if rob < 1.0:
        conf = 0.5 + (conf - 0.5) * rob
    return max(0.0, min(1.0, conf))


scenarios = [
    {"name": "Strong support", "cov": 1.0, "sup": 1.0, "contra": 0.0, "miss": 0.0, "spec": 0.9, "cons": 1.0, "rob": 0.95},
    {"name": "Mixed evidence", "cov": 0.6, "sup": 0.6, "contra": 0.3, "miss": 0.2, "spec": 0.6, "cons": 0.7, "rob": 0.7},
    {"name": "Mostly missing", "cov": 0.2, "sup": 0.2, "contra": 0.0, "miss": 0.8, "spec": 0.3, "cons": 1.0, "rob": 0.9},
    {"name": "Contradiction", "cov": 0.4, "sup": 0.3, "contra": 0.8, "miss": 0.6, "spec": 0.7, "cons": 0.4, "rob": 0.9},
]

for s in scenarios:
    conf = confidence_score(
        cov=s["cov"],
        sup=s["sup"],
        contra=s["contra"],
        miss_ratio=s["miss"],
        spec=s["spec"],
        cons=s["cons"],
        rob=s["rob"],
    )
    print(f"{s['name']:<16} -> confidence={conf:.2f}")


## Chunk Impact and Robustness Approximation (Reference)

The following helper functions mirror the production logic for robustness approximation and chunk impact scoring. They are deterministic and require only chunk-item labels, supporting the "no extra LLM calls" requirement.


In [ ]:
from collections import defaultdict


def robustness_approx(labels, item_ids):
    # labels: list of {item_id, chunk_id, relation}
    support_chunks_by_item = {rid: set() for rid in item_ids}
    contra_items = set()
    support_labels = 0
    weak_labels = 0

    for lbl in labels:
        rid = lbl["item_id"]
        if rid not in support_chunks_by_item:
            continue
        if lbl["relation"] in {"support", "weak"}:
            support_chunks_by_item[rid].add(lbl["chunk_id"])
            support_labels += 1
            if lbl["relation"] == "weak":
                weak_labels += 1
        elif lbl["relation"] == "contradict":
            contra_items.add(rid)

    supported_items = [rid for rid, chunks in support_chunks_by_item.items() if chunks]
    supported_count = len(supported_items)
    single_support_ratio = (
        sum(1 for rid in supported_items if len(support_chunks_by_item[rid]) == 1)
        / max(1, supported_count)
    )
    weak_ratio = weak_labels / max(1, support_labels)
    contradiction_ratio = len(contra_items) / max(1, len(item_ids))
    redundancy_mean = (
        sum(len(support_chunks_by_item[rid]) for rid in supported_items) / max(1, supported_count)
    )

    # penalty → robustness
    penalty = 0.35 * single_support_ratio + 0.2 * weak_ratio + 0.2 * contradiction_ratio
    penalty += 0.1 * max(0.0, 1.0 - min(1.0, (redundancy_mean - 1.0) / 2.0))
    rob = max(0.0, min(1.0, 1.0 - penalty))
    return rob


def chunk_impact(labels, item_weights=None, specificity_by_label=None, directness_by_label=None):
    # labels: list of {item_id, chunk_id, relation, specificity, directness}
    if item_weights is None:
        item_weights = defaultdict(lambda: 1.0)
    support_chunks = defaultdict(set)
    labels_by_chunk = defaultdict(list)

    for lbl in labels:
        if lbl["relation"] not in {"support", "weak"}:
            continue
        support_chunks[lbl["item_id"]].add(lbl["chunk_id"])
        labels_by_chunk[lbl["chunk_id"]].append(lbl)

    n_by_item = {item_id: len(chunks) for item_id, chunks in support_chunks.items() if chunks}
    raw_scores = {}
    for chunk_id, lbls in labels_by_chunk.items():
        items_supported = {l["item_id"] for l in lbls}
        base_credit = sum(item_weights[i] / max(1, n_by_item.get(i, 0)) for i in items_supported)
        spec = max(l.get("specificity", 1.0) for l in lbls)
        direct = max(l.get("directness", 1.0) for l in lbls)
        raw_scores[chunk_id] = base_credit * spec * direct

    total = sum(raw_scores.values())
    if total <= 0:
        return {cid: 0.0 for cid in raw_scores}
    return {cid: val / total for cid, val in raw_scores.items()}


example_labels = [
    {"item_id": "item_1", "chunk_id": "C1", "relation": "support", "specificity": 1.0, "directness": 1.0},
    {"item_id": "item_1", "chunk_id": "C2", "relation": "support", "specificity": 0.6, "directness": 1.0},
    {"item_id": "item_2", "chunk_id": "C2", "relation": "weak", "specificity": 0.6, "directness": 0.5},
]

print("ROB approx:", robustness_approx(example_labels, ["item_1", "item_2"]))
print("Impact:", chunk_impact(example_labels))


## Limitations and Improvement Opportunities

### Current limitations

- **Label quality is the main bottleneck**. The downstream posterior, confidence score, robustness estimate, and chunk impact values are only as reliable as the chunk-to-item labels emitted during evaluation.
- **Rubric decomposition is still heuristic**. When a requirement is split into rubric items using simple textual rules, the scoring layer may inherit structural errors from that decomposition.
- **Robustness approximation is informative, but not equivalent to a true perturbation study**. The `approx` mode captures useful structural warning signs, yet it does not directly observe whether a re-labeled chunking variant would change the conclusion.
- **Chunk impact is an attribution heuristic, not causal inference**. The score reflects evidence contribution under a shared-credit model; it should not be interpreted as a proof that removing a chunk would change the finding.
- **Version 1 does not model polarity-aware impact separately**. A single scalar impact score is easier to explain operationally, but it does not distinguish supportive influence from contradictory influence.
- **Uniform item weighting is a simplifying assumption**. Some requirements contain items that are materially more important than others, and equal weighting may understate that difference.
- **`max_chunks` introduces a practical bias-coverage trade-off**. Restricting the number of evaluated chunks improves cost and latency, but may exclude relevant evidence from the scoring layer.
- **Directness is only partially operationalized**. When directness is absent or weakly calibrated, specificity may dominate the score even when the evidence is only indirectly relevant.
- **Confidence weights are not yet empirically calibrated**. The current weighted logistic formulation is a sound starting point, but its coefficients should ultimately be fitted against reviewed historical audit outcomes.

### Recommended improvements

- Build a reviewed benchmark set and calibrate both conformance probability and confidence against expert judgments.
- Replace heuristic rubric decomposition with either reviewer-authored rubric items or a stronger structured decomposition step.
- Introduce requirement-specific or domain-specific item weights where controls are not equally material.
- Add selective counterfactual validation on a sampled subset to compare approximate chunk impact against leave-one-out re-evaluation.
- Extend a future version with separate **supporting impact** and **contradicting impact** channels when polarity becomes analytically important.
- Strengthen directness modeling so that contextual support, documentary support, and explicit testable evidence are clearly separated.
- Perform sensitivity analysis over chunking strategy, overlap, and `max_chunks` so that the scoring layer can be characterized under realistic operational variation.
- Consider confidence intervals or uncertainty bands for the derived metrics once sufficient calibration data is available.

These limitations do not invalidate the methodology; rather, they define the boundary between an explainable, production-practical scoring layer and a fully calibrated causal evidence model.


## Calibration and Validation

1. Collect a historical set of evaluated requirements with human-reviewed conformance labels.
2. Compute the confidence features for each finding using the same chunk-labeling logic.
3. Fit the confidence weights (logistic regression or isotonic calibration) against correctness of the LLM judgment.
4. Re-evaluate weights periodically to prevent drift, especially when chunking or prompts change.

**Key metrics**:
- Accuracy of conformance classification
- Calibration (e.g., reliability curve)
- Robustness sensitivity (confidence should fall when perturbations flip conclusions)
- Impact stability (chunk impact should not be dominated by single-token boundary effects)

This methodology keeps the evaluation decision interpretable and separates *what the evidence implies* from *how sure we are*.
